In [1]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import pandas as pd

In [2]:
# 데이터 준비
X_scaled = pd.read_csv('single_variable_data.csv')

In [3]:
# 테스트할 eps 값들 (0.30 대신)
for eps in [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]:
    dbscan = DBSCAN(eps=eps, min_samples=5)
    labels = dbscan.fit_predict(X_scaled)
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    noise_ratio = n_noise / len(labels)
    
    print(f"eps={eps:.1f}: clusters={n_clusters}, noise={noise_ratio:.1%}")
    
    if 0.05 < noise_ratio < 0.3 and n_clusters >= 2:
        print(f"  ✅ 좋음! 이 값을 사용하세요.")
        # 이 파라미터로 최종 실행
        final_labels = labels
        break

eps=1.0: clusters=12, noise=2.2%
eps=1.5: clusters=2, noise=0.0%
eps=2.0: clusters=1, noise=0.0%
eps=2.5: clusters=1, noise=0.0%
eps=3.0: clusters=1, noise=0.0%
eps=4.0: clusters=1, noise=0.0%
eps=5.0: clusters=1, noise=0.0%


In [4]:
from sklearn.metrics import silhouette_score
import pandas as pd
import numpy as np
from collections import Counter

In [5]:
# 최적 파라미터로 DBSCAN 실행
dbscan = DBSCAN(eps=1.0, min_samples=7)  # min_samples는 7로 유지
labels = dbscan.fit_predict(X_scaled)


In [6]:
# ============================================
# 상세 분석
# ============================================
print("="*60)
print("DBSCAN 최종 결과 분석 (eps=1.0, min_samples=7)")
print("="*60)

DBSCAN 최종 결과 분석 (eps=1.0, min_samples=7)


In [7]:
# 기본 통계
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)
n_total = len(labels)

print(f"\n📊 기본 통계:")
print(f"  총 데이터: {n_total}개")
print(f"  클러스터 개수: {n_clusters}개")
print(f"  노이즈: {n_noise}개 ({n_noise/n_total:.1%})")



📊 기본 통계:
  총 데이터: 3448개
  클러스터 개수: 10개
  노이즈: 96개 (2.8%)


In [8]:
# 클러스터별 크기
cluster_counts = Counter(labels)
print(f"\n📦 클러스터별 크기:")
for cluster_id, count in sorted(cluster_counts.items()):
    if cluster_id == -1:
        print(f"  Noise: {count}개 ({count/n_total:.1%})")
    else:
        print(f"  Cluster {cluster_id:2d}: {count:4d}개 ({count/n_total:.1%})")



📦 클러스터별 크기:
  Noise: 96개 (2.8%)
  Cluster  0: 1688개 (49.0%)
  Cluster  1:  298개 (8.6%)
  Cluster  2:   93개 (2.7%)
  Cluster  3: 1048개 (30.4%)
  Cluster  4:   66개 (1.9%)
  Cluster  5:   92개 (2.7%)
  Cluster  6:   22개 (0.6%)
  Cluster  7:   17개 (0.5%)
  Cluster  8:   20개 (0.6%)
  Cluster  9:    8개 (0.2%)


In [9]:
# 실루엣 스코어 (노이즈 제외)
mask = labels != -1
if len(set(labels[mask])) >= 2:
    sil_score = silhouette_score(X_scaled[mask], labels[mask])
    print(f"\n🎯 실루엣 스코어 (노이즈 제외): {sil_score:.4f}")
    
    if sil_score > 0.5:
        print("   평가: ✅ 강한 클러스터 구조")
    elif sil_score > 0.3:
        print("   평가: ⚠️ 적절한 클러스터 구조")
    else:
        print("   평가: ❌ 약한 클러스터 구조")


🎯 실루엣 스코어 (노이즈 제외): 0.1712
   평가: ❌ 약한 클러스터 구조
